# Autoregressive Transformer

The transformer models the joint distribution over codebook token sequences autoregressively: given the class label and all previous tokens, predict the next. At generation time, tokens are sampled one by one left-to-right, top-to-bottom, then decoded by the VQ-VAE decoder.

In [1]:
import torch
import torch.nn as nn
import math

In [2]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'using {device}')

using mps


## 2D Positional Encoding

Tokens come from an 8×8 spatial grid — a flat index carries no spatial meaning on its own. We decompose each position into its row and column and encode each independently with sinusoidal embeddings, then concatenate. This gives the transformer a sense of both axes without conflating them.

In [3]:
def positional_encoding(pos, n_cols, d_model):
    d = d_model // 2
    half = d // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half) / half)  # CPU
    row_pos = pos // n_cols
    col_pos = pos % n_cols
    row_args = row_pos.float().view(-1, 1) @ freqs.view(1, -1)
    col_args = col_pos.float().view(-1, 1) @ freqs.view(1, -1)
    return torch.cat([row_args.sin(), row_args.cos(), col_args.sin(), col_args.cos()], dim=1)

## GPT

The label is prepended as the first token via a learned embedding. A causal (lower-triangular) mask ensures each position can only attend to earlier ones. The transformer predicts a distribution over the `K`-entry codebook at each position; during training the target at position `i` is the actual token at position `i+1`.

In [4]:
class GPT(nn.Module):
    def __init__(self, K=512, S=8, d_model=256, nhead=8, num_layers=8):
        super().__init__()
        self.K = K
        self.token_emb = nn.Embedding(K, d_model)
        self.label_emb = nn.Embedding(10, d_model)
        pos_encoding = positional_encoding(torch.arange(0, S*S - 1), S, d_model)  # (S*S-1, d)
        self.register_buffer('pos_encoding', pos_encoding)
        self.register_buffer('mask', torch.tril(torch.ones(S*S, S*S, dtype=torch.bool)))  # (49, 49)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.linear = nn.Linear(d_model, K)

    def forward(self, seq):
        l_emb = self.label_emb(seq[:, :1])       # (B, 1, d)
        t_emb = self.token_emb(seq[:, 1:])       # (B, S*S-1, d)
        t_emb = t_emb + self.pos_encoding        # (S*S-1, d) — no slicing needed
        x    = torch.cat([l_emb, t_emb], dim=1)  # (B, S*S, d)
        out  = self.transformer(x, mask=self.mask, is_causal=True)
        return self.linear(out)                   # (B, S*S, K)

## Sanity Check

Input is a sequence of length `S²+1` (1 label + 64 tokens for an 8×8 grid). The transformer outputs logits of shape `(B, S²+1, K)`.

In [5]:
model = GPT().to(device)
x = torch.randint(0, 512, (4, 65), device=device)
out = model.forward(x[:, :-1])
print(out.shape)  # expect (4, 64, 512)